# ResikIn Waste Classifier - Model Fine-Tuning

Notebook ini digunakan untuk melatih (fine-tune) model CLIP agar lebih pintar mendeteksi sampah. Kita akan:
1. Mengunduh dataset dari Roboflow langsung ke server Google Colab.
2. Menjalankan skrip persiapan data.
3. Menjalankan proses training.

## 1. Persiapan Lingkungan (Install Dependencies)

In [7]:
!pip install -q roboflow transformers torch torchvision scikit-learn pillow matplotlib tqdm

## 2. Unduh Dataset dari Roboflow

Sesuai dengan API yang Anda berikan, ini akan mengunduh dataset ke dalam folder `dataset_mentah` di Colab.

In [8]:
import os
from roboflow import Roboflow

# Buat folder untuk dataset mentah
os.makedirs("dataset_mentah", exist_ok=True)
os.chdir("dataset_mentah")

# Download dari Roboflow menggunakan key Anda
rf = Roboflow(api_key="CsZFCzVXJL8qqYDhx0hR")
project = rf.workspace("project-ia-andzk").project("classification-image-6zihm")
version = project.version(2)
dataset = version.download("folder")

print(f"\nDataset berhasil diunduh di path: {dataset.location}")
os.chdir("..")

loading Roboflow workspace...
loading Roboflow project...

Dataset berhasil diunduh di path: /content/dataset_mentah/classification-image-2


## 3. Clone Repository Anda

Kita perlu mengunduh skrip `train.py` dan `prepare_dataset.py` yang sudah dibuat. Pastikan Anda sudah me-*push* kode ke repositori GitHub Anda.

In [9]:
# Ganti URL ini dengan URL repository GitHub baru Anda yang berisi resikin-waste-classifier
# Jika repo private, hapus baris ini dan unggah file secara manual.
!git clone https://github.com/Hanafi-Sh/resikin-ai.git repo_ai

# Jika Anda belum push repo baru, Anda bisa mengunggah folder resikin-waste-classifier
# secara manual ke Colab dan mengubah path di sel-sel berikutnya.

Cloning into 'repo_ai'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 22 (delta 0), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 12.37 KiB | 12.37 MiB/s, done.


## 4. Rapikan Dataset

Roboflow mengunduh dataset dengan nama folder tertentu. Kita akan menggunakan skrip `prepare_dataset.py` untuk memindahkannya ke dalam struktur yang diharapkan oleh skrip training.

In [10]:
import glob

# Mencari folder hasil download (nama foldernya biasanya sama dengan nama project)
download_dirs = glob.glob("dataset_mentah/*/")
roboflow_dir = download_dirs[0] if download_dirs else ""

if roboflow_dir:
    print(f"Menggunakan direktori Roboflow: {roboflow_dir}")
    !python repo_ai/src/preprocessing/prepare_dataset.py --roboflow_dir "{roboflow_dir}" --output_dir "./data"
else:
    print("Folder dataset tidak ditemukan!")

Menggunakan direktori Roboflow: dataset_mentah/classification-image-2/
🔄 Organizing Roboflow dataset...
  [train] vide: 167 images
  [train] pleine: 126 images
⚠️  Split 'valid' not found, skipping...
⚠️  Split 'test' not found, skipping...

✅ Dataset siap digunakan untuk training!

📊 Dataset Statistics:
----------------------------------------
  train  / pleine          :   126 images
  train  / vide            :   167 images


## 5. Mulai Training! 🚀

Ini akan melatih model CLIP. Waktu yang dibutuhkan sekitar 20-40 menit tergantung jumlah dataset dan GPU (pastikan Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**).

In [11]:
!python repo_ai/scripts/train.py --data_dir "./data" --output_dir "./models" --epochs 10 --batch_size 32

Device: cuda
Loading CLIP ViT-B/32...
config.json: 4.19kB [00:00, 12.3MB/s]
pytorch_model.bin: 100% 605M/605M [00:03<00:00, 186MB/s]
Loading weights:  73% 291/398 [00:00<00:00, 1036.38it/s, Materializing param=vision_model.encoder.layers.5.self_attn.k_proj.weight]
Loading weights: 100% 398/398 [00:00<00:00, 888.11it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

model.safetensors:   0% 0.00/605M [00:00<?, ?B/s]
preprocessor_config.json: 100% 316/316 [00:00<00:00, 1.09MB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model chec

## 6. Unduh Hasil Model

Setelah training selesai, kita akan mengompres folder `models/` agar Anda bisa mengunduhnya ke laptop Anda.

In [12]:
import shutil
from google.colab import files

# Zip folder models
shutil.make_archive("hasil_model_clip", 'zip', "./models")

# Download zip file
files.download("hasil_model_clip.zip")

FileNotFoundError: [Errno 2] No such file or directory: './models'